# Credit Card Default Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether a credit card holder will **default on payment** next month based on demographics, credit limit, payment history, and bill amounts. ~30,000 rows with ~22% default rate.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a credit card customer's payment history, billing amounts, and demographics, predict whether they will default on their next payment.

## 5 · Why This Project Matters

- Credit card defaults cost banks billions in losses annually.
- Accurate default prediction enables **risk-based pricing** and **credit limit management**.
- This dataset teaches handling of **temporal payment features** and **class imbalance**.
- Financial classification requires **calibrated probabilities**, not just labels.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~30,000 |
| Features | 23 (demographics + 6-month payment history) |
| Target | `default` (binary: 0/1) |
| Class balance | ~78% No default, ~22% Default |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: UCI ML Repository — Default of Credit Card Clients dataset.
**License**: Public / educational.
**Citation**: Yeh & Lien, 2009.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "default"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '.'
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Credit Card Default Prediction


## 11 · Dataset Download or Loading

In [4]:
# Generate synthetic credit card default data (inspired by UCI dataset)
np.random.seed(SEED)
n = 30000

LIMIT_BAL = np.random.choice([10000,20000,30000,50000,80000,100000,150000,200000,300000,500000], n)
SEX = np.random.choice([1, 2], n)  # 1=male, 2=female
EDUCATION = np.random.choice([1,2,3,4], n, p=[0.1,0.5,0.3,0.1])
MARRIAGE = np.random.choice([1,2,3], n, p=[0.35,0.55,0.1])
AGE = np.random.randint(21, 75, n)

# Payment status: -1=pay duly, 1=delay 1mo, 2=delay 2mo, etc.
PAY_0 = np.random.choice([-1,0,1,2,3,4,5], n, p=[0.3,0.3,0.15,0.1,0.08,0.05,0.02])
PAY_2 = np.random.choice([-1,0,1,2,3], n, p=[0.35,0.35,0.15,0.1,0.05])
PAY_3 = np.random.choice([-1,0,1,2,3], n, p=[0.35,0.35,0.15,0.1,0.05])
PAY_4 = np.random.choice([-1,0,1,2], n, p=[0.4,0.35,0.15,0.1])
PAY_5 = np.random.choice([-1,0,1,2], n, p=[0.4,0.35,0.15,0.1])
PAY_6 = np.random.choice([-1,0,1,2], n, p=[0.4,0.35,0.15,0.1])

BILL_AMT1 = (LIMIT_BAL * np.random.beta(2,5,n)).astype(int)
BILL_AMT2 = (LIMIT_BAL * np.random.beta(2,5,n)).astype(int)
BILL_AMT3 = (LIMIT_BAL * np.random.beta(2,5,n)).astype(int)
PAY_AMT1 = (BILL_AMT1 * np.random.beta(3,3,n) + 100).astype(int)
PAY_AMT2 = (BILL_AMT2 * np.random.beta(3,3,n) + 100).astype(int)
PAY_AMT3 = (BILL_AMT3 * np.random.beta(3,3,n) + 100).astype(int)

score = (
    0.3 * (PAY_0 > 0).astype(float)
    + 0.2 * (PAY_2 > 0).astype(float)
    + 0.1 * (PAY_3 > 0).astype(float)
    - 0.1 * np.log1p(LIMIT_BAL) / 13
    + 0.05 * (BILL_AMT1 / (LIMIT_BAL + 1))
    + np.random.normal(0, 0.2, n)
)
default_val = (score > 0.25).astype(int)

df = pd.DataFrame({
    'LIMIT_BAL': LIMIT_BAL, 'SEX': SEX, 'EDUCATION': EDUCATION,
    'MARRIAGE': MARRIAGE, 'AGE': AGE,
    'PAY_0': PAY_0, 'PAY_2': PAY_2, 'PAY_3': PAY_3,
    'PAY_4': PAY_4, 'PAY_5': PAY_5, 'PAY_6': PAY_6,
    'BILL_AMT1': BILL_AMT1, 'BILL_AMT2': BILL_AMT2, 'BILL_AMT3': BILL_AMT3,
    'PAY_AMT1': PAY_AMT1, 'PAY_AMT2': PAY_AMT2, 'PAY_AMT3': PAY_AMT3,
    'default': default_val,
})
print(f"Dataset shape: {df.shape}")
print(f"Default rate: {df['default'].mean():.2%}")
df.head()

Dataset shape: (30000, 18)
Default rate: 33.93%


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,PAY_AMT1,PAY_AMT2,PAY_AMT3,default
0,150000,1,3,3,47,0,1,0,-1,0,1,8058,18226,21843,3055,10515,14534,0
1,50000,2,2,2,53,0,3,2,-1,0,-1,19796,3687,8761,6814,2314,3183,0
2,200000,1,1,3,36,0,-1,0,0,-1,0,107243,95390,12962,19180,69610,6199,0
3,80000,1,2,1,27,0,0,1,0,2,-1,53039,45650,8621,30676,17460,4015,0
4,150000,1,2,3,33,5,2,-1,-1,-1,2,36323,42928,58275,17106,24700,33302,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (30000, 18)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
default
0    19821
1    10179
Name: count, dtype: int64

Dtypes:
LIMIT_BAL    int64
SEX          int64
EDUCATION    int64
MARRIAGE     int64
AGE          int32
PAY_0        int64
PAY_2        int64
PAY_3        int64
PAY_4        int64
PAY_5        int64
PAY_6        int64
BILL_AMT1    int64
BILL_AMT2    int64
BILL_AMT3    int64
PAY_AMT1     int64
PAY_AMT2     int64
PAY_AMT3     int64
default      int64
dtype: object

Target 'default' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols[:6]):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=10)
plt.suptitle("Feature Distributions (first 6)", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

# Correlation with target
corr_target = df[num_cols].corrwith(df[TARGET]).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
corr_target.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title("Feature Correlation with Default")
plt.tight_layout()
plt.show()
print(f"Features: {len(num_cols)}")

Features: 17


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Default Distribution")
axes[0].set_xticklabels(['No Default (0)', 'Default (1)'], rotation=0)

df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions")
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / df[TARGET].value_counts().iloc[1]:.2f}:1")

Class distribution:
default
0    19821
1    10179
Name: count, dtype: int64

Imbalance ratio: 1.95:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (24000, 17), Test: (6000, 17)
Train target dist: {0: 15857, 1: 8143}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean[:10]}...")

Features (17): ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5']...


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.7330
Precision: 0.7225
Recall   : 0.7330
F1       : 0.7213
ROC-AUC  : 0.7809


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.728333           0.714133  0.783777  0.732164   0.739290  0.728333    0.051341
BernoulliNB                    0.756167           0.711070  0.824202  0.751222   0.749959  0.756167    0.069693
RandomForestClassifier         0.754500           0.706942  0.813680  0.748730   0.747706  0.754500    4.881026
CatBoostClassifier             0.752500           0.701845  0.819066  0.745627   0.745051  0.752500    6.688740
LGBMClassifier                 0.756500           0.701647  0.820371  0.747937   0.748834  0.756500    0.271852
AdaBoostClassifier             0.762333           0.698059  0.826133  0.749665   0.755962  0.762333    1.844121
XGBClassifier                  0.740667           0.697189  0.804550  

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: rf
Accuracy : 0.7583
F1       : 0.7491


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

# Add baseline and FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.7593  F1: 0.7485


LightGBM  Acc: 0.7508  F1: 0.7455


XGBoost   Acc: 0.7505  F1: 0.7451


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.7583  0.7491     0.7508  0.7583
CatBoost               0.7593  0.7485     0.7520  0.7593
LightGBM               0.7508  0.7455     0.7442  0.7508
XGBoost                0.7505  0.7451     0.7438  0.7505
Logistic Regression    0.7330  0.7213     0.7225  0.7330

Top 3: ['FLAML', 'CatBoost', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  FLAML
              precision    recall  f1-score   support

           0       0.78      0.88      0.83      3964
           1       0.69      0.53      0.60      2036

    accuracy                           0.76      6000
   macro avg       0.74      0.70      0.71      6000
weighted avg       0.75      0.76      0.75      6000


  CatBoost
              precision    recall  f1-score   support

           0       0.78      0.89      0.83      3964
           1       0.70      0.51      0.59      2036

    accuracy                           0.76      6000
   macro avg       0.74      0.70      0.71      6000
weighted avg       0.75      0.76      0.75      6000


  LightGBM
              precision    recall  f1-score   support

           0       0.79      0.85      0.82      3964
           1       0.66      0.56      0.60      2036

    accuracy                           0.75      6000
   macro avg       0.72      0.70      0.71      6000

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")

# Errors by class
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: FLAML
Error rate: 0.2417 (1450 / 6000)

Errors by true class:
      errors  total  error_rate
true                           
0        486   3964    0.122603
1        964   2036    0.473477


## 25 · Interpretation and Business Insight

- **Payment status history** (PAY_0 to PAY_6) is the strongest predictor group.
- Recent payment delays (PAY_0) are more predictive than older ones.
- **Credit limit (LIMIT_BAL)** has a negative correlation with default.
- Bill amounts and payment amounts provide utilization signals.
- Demographic features (education, marriage, age) have weak but non-zero effects.

## 26 · Limitations

1. Taiwan-specific data — credit behavior differs by country.
2. 2005 data — banking products and behavior have evolved.
3. ~22% default rate — moderate imbalance.
4. No macro-economic context (GDP, unemployment).
5. Payment status encoding is non-standard (requires domain understanding).

## 27 · How to Improve This Project

1. Engineer utilization ratios (bill/limit) and payment ratios.
2. Apply class weights or SMOTE for better default recall.
3. Calibrate probabilities for risk scoring.
4. Add macro-economic features for temporal context.
5. Use PR-AUC as the primary metric (imbalanced data).

## 28 · Production Considerations

- Monthly default risk scoring for credit portfolio management.
- Integration with credit limit adjustment systems.
- Regulatory compliance (Basel III/IV capital requirements).
- Model validation and stress testing frameworks.

## 29 · Common Mistakes

1. Using accuracy on imbalanced credit data.
2. Not calibrating probabilities for risk-based decisions.
3. Ignoring the sequential nature of payment history.
4. Treating PAY_0 categories as continuous.
5. Not considering regulatory requirements for model explainability.

## 30 · Mini Challenge / Exercises

1. Engineer a 'payment_ratio' feature (payment/bill) and measure impact.
2. Build a model using only PAY_0 — how close to the full model?
3. Calibrate the best model's probabilities and plot reliability curve.
4. Compare PR-AUC across all models — which is best for default detection?

## 31 · Final Summary / Key Takeaways

1. Credit default prediction is a core banking ML application.
2. Payment history (especially recent) dominates prediction.
3. PR-AUC and recall matter more than accuracy for defaults.
4. Probability calibration is essential for risk-based decisions.
5. Regulatory compliance requires model transparency and validation.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }

with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.7593,
    "f1": 0.7485,
    "precision": 0.752,
    "recall": 0.7593
  },
  "LightGBM": {
    "accuracy": 0.7508,
    "f1": 0.7455,
    "precision": 0.7442,
    "recall": 0.7508
  },
  "XGBoost": {
    "accuracy": 0.7505,
    "f1": 0.7451,
    "precision": 0.7438,
    "recall": 0.7505
  },
  "Logistic Regression": {
    "accuracy": 0.733,
    "f1": 0.7213,
    "precision": 0.7225,
    "recall": 0.733
  },
  "FLAML": {
    "accuracy": 0.7583,
    "f1": 0.7491,
    "precision": 0.7508,
    "recall": 0.7583
  }
}
